# Traducción automática de aspectos (Paso 1 del enfoque híbrido) — versión definitiva

Traduce cada valor distinto de `aspecto` en `gold.nlp_aspectos_resenas` a español, y los guarda en `gold.aspecto_traducciones`.

**Ejecutar en local, no en Colab** -- las máquinas de Colab comparten IP de salida entre muchos usuarios, lo que agota mucho más rápido el límite de peticiones del traductor gratuito.

**Traductor híbrido**: intenta primero Google (mejor calidad cuando funciona), y si falla o está bloqueado, cae automáticamente a MyMemory con detección de idioma de origen -- así el proceso avanza aunque uno de los dos servicios esté temporalmente bloqueado.

**Diseño incremental y resistente a cortes**: procesa por orden de frecuencia, guarda cada 20 según traduce, se puede interrumpir y retomar sin perder nada.

## Paso 1 — Instalar dependencias

In [1]:
%pip install -q deep-translator langdetect sqlalchemy psycopg2-binary python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## Paso 2 — Conectar

**Usa `AZURE_DB_URL_R`**, no `AZURE_DB_URL` (esa última apunta al servidor viejo, ya inalcanzable tras la migración).

In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()
engine = create_engine(os.environ['AZURE_DB_URL_R'], pool_pre_ping=True, pool_recycle=280)

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. PostGIS:', version)
print('Host:', os.environ['AZURE_DB_URL_R'].split('@')[1].split('/')[0] if '@' in os.environ['AZURE_DB_URL_R'] else '(revisar formato)')

Conectado. PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1
Host: db-tfm-tenerife2.postgres.database.azure.com:5432


## Paso 3 — Obtener los aspectos distintos aún no traducidos, por orden de frecuencia

In [3]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS gold.aspecto_traducciones (
            aspecto_original text PRIMARY KEY,
            aspecto_traducido text,
            frecuencia integer
        )
    '''))
    conn.execute(text('''
        ALTER TABLE gold.aspecto_traducciones
        ADD COLUMN IF NOT EXISTS motor text
    '''))
print('Tabla gold.aspecto_traducciones lista (creada/actualizada si hacia falta).')

df_pendientes = pd.read_sql('''
    SELECT a.aspecto, COUNT(*) AS frecuencia
    FROM gold.nlp_aspectos_resenas a
    WHERE a.aspecto IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 FROM gold.aspecto_traducciones t WHERE t.aspecto_original = a.aspecto
      )
    GROUP BY a.aspecto
    ORDER BY frecuencia DESC
''', engine)

print('Aspectos distintos pendientes de traducir:', len(df_pendientes))
print(df_pendientes.head(10))

Tabla gold.aspecto_traducciones lista (creada/actualizada si hacia falta).
Aspectos distintos pendientes de traducir: 6668
                 aspecto  frecuencia
0            kit de baño           1
1             plataforma           1
2       al fresco dining           1
3             światłowód           1
4             can opener           1
5               fiambres           1
6                   dort           1
7                  kedel           1
8                  петух           1
9  kosmetyków do kąpieli           1


## Paso 3b — Limpiar traducciones contaminadas (solo si hace falta)

Ejecuta esto únicamente si sabes que hay filas guardadas con el texto de una página de error en vez de una traducción real.

In [4]:
with engine.begin() as conn:
    resultado = conn.execute(text('''
        DELETE FROM gold.aspecto_traducciones
        WHERE LENGTH(aspecto_traducido) > 200
           OR aspecto_traducido ILIKE '%error 500%'
           OR aspecto_traducido ILIKE '%server error%'
           OR aspecto_traducido ILIKE '%that''s an error%'
           OR aspecto_traducido ILIKE '%please try again later%'
    '''))
print('Filas contaminadas borradas:', resultado.rowcount)
print('Vuelve a ejecutar el Paso 3 para actualizar la lista de pendientes.')

Filas contaminadas borradas: 0
Vuelve a ejecutar el Paso 3 para actualizar la lista de pendientes.


## Paso 4a — Preparar el traductor híbrido (Google primero, MyMemory como respaldo)

MyMemory exige códigos de idioma completos (`es-ES`, no `es`) y no acepta `'auto'` como origen -- por eso hace falta detectar el idioma con `langdetect` antes de usarlo, y traducir ese código corto al formato que MyMemory espera. En vez de mapear cada idioma a mano, se construye el mapa automáticamente a partir del propio diccionario de idiomas soportados por MyMemory.

In [5]:
import time
from deep_translator import GoogleTranslator, MyMemoryTranslator
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # resultados deterministas

MARCADORES_ERROR = ['error 500', 'server error', "that's an error", 'please try again later', "that's all we know"]

MI_EMAIL = 'guilleproducer@gmail.com'  # sube el limite diario gratuito de MyMemory


def es_traduccion_valida(texto):
    if not texto:
        return False
    if len(texto) > 200:
        return False
    texto_lower = texto.lower()
    return not any(marcador in texto_lower for marcador in MARCADORES_ERROR)


# Mapa dinamico: codigo ISO de 2 letras (lo que da langdetect) -> codigo completo de MyMemory
_dummy = MyMemoryTranslator(source='en-GB', target='es-ES')
_tabla_idiomas_mymemory = _dummy._languages

MAPA_ISO_A_MYMEMORY = {}
for _nombre, _codigo in _tabla_idiomas_mymemory.items():
    _iso = _codigo.split('-')[0].lower()
    if _iso not in MAPA_ISO_A_MYMEMORY:
        MAPA_ISO_A_MYMEMORY[_iso] = _codigo

# Overrides para las variantes mas comunes en turismo europeo
MAPA_ISO_A_MYMEMORY.update({'en': 'en-GB', 'es': 'es-ES', 'pt': 'pt-PT', 'fr': 'fr-FR'})


def detectar_idioma_mymemory(texto):
    try:
        iso = detect(texto).split('-')[0].lower()
    except Exception:
        iso = 'en'
    return MAPA_ISO_A_MYMEMORY.get(iso, 'en-GB')


def traducir_con_reintentos(texto, intentos_max_mymemory=2):
    # 1. Intento rapido con Google (una sola vez, sin reintentos largos --
    #    si esta bloqueado, no merece la pena esperar minutos por termino)
    try:
        resultado = GoogleTranslator(source='auto', target='es').translate(texto)
        if es_traduccion_valida(resultado):
            return resultado, 'google'
    except Exception:
        pass

    # 2. Respaldo: MyMemory con el idioma detectado y el email (sube el limite diario)
    idioma_mm = detectar_idioma_mymemory(texto)
    espera = 2
    for intento in range(intentos_max_mymemory):
        try:
            resultado = MyMemoryTranslator(source=idioma_mm, target='es-ES', email=MI_EMAIL).translate(texto)
            if es_traduccion_valida(resultado):
                return resultado, 'mymemory'
            print(f'    respuesta invalida en mymemory para "{texto}"')
        except Exception as error:
            print(f'    error en mymemory para "{texto}" (intento {intento + 1}/{intentos_max_mymemory}): {error}')
        time.sleep(espera)
        espera *= 2

    return None, None


print('Traductor hibrido listo, con email configurado.')

Traductor hibrido listo, con email configurado.


## Paso 4b — Prueba rápida antes de lanzar todo

In [6]:
for palabra in ['tapones', 'cleaning lady', 'remote control', 'posizione', 'lage']:
    traducido, motor = traducir_con_reintentos(palabra)
    print(palabra, '->', traducido, f'(via {motor})')

    error en mymemory para "tapones" (intento 1/2): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
    error en mymemory para "tapones" (intento 2/2): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
tapones -> None (via None)
    error en mymemory para "cleaning lady" (intento 1/2): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
    error en mymemory para "cleaning lady" (intento 2/2): Server Error: You made too many requests to the server.Accordin

## Paso 5 — Traducir por bloques, guardando según se avanza

In [7]:
GUARDAR_CADA = 20

buffer = []
procesados = 0
total = len(df_pendientes)
inicio = time.time()

for _, fila in df_pendientes.iterrows():
    original = fila['aspecto']
    traducido, motor = traducir_con_reintentos(original)
    traducido_normalizado = traducido.strip().lower() if traducido else None

    buffer.append({
        'aspecto_original': original,
        'aspecto_traducido': traducido_normalizado,
        'frecuencia': int(fila['frecuencia']),
        'motor': motor,
    })
    procesados += 1
    time.sleep(0.5)

    if len(buffer) >= GUARDAR_CADA or procesados == total:
        df_buffer = pd.DataFrame(buffer)
        df_buffer.to_sql('aspecto_traducciones', engine, schema='gold', if_exists='append', index=False)
        buffer = []

        transcurrido = time.time() - inicio
        velocidad = procesados / transcurrido if transcurrido > 0 else 0
        restantes = total - procesados
        eta_min = (restantes / velocidad / 60) if velocidad > 0 else 0
        print(f'Guardados {procesados}/{total} -- {velocidad:.1f} aspectos/seg -- estimado restante: {eta_min:.0f} min')

print()
print('Completado. Total traducido en esta ejecucion:', procesados)

    error en mymemory para "plataforma" (intento 1/2): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
    error en mymemory para "plataforma" (intento 2/2): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
    error en mymemory para "al fresco dining" (intento 1/2): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
    error en mymemory para "al fresco dining" (intento 2/2): Server Error: You made too many requests to the server.According to google, yo

KeyboardInterrupt: 

## Paso 6 — Revisar qué se ha agrupado, y qué motor se usó

In [8]:
with engine.connect() as conn:
    resumen_motor = pd.read_sql('''
        SELECT motor, COUNT(*) AS num_terminos
        FROM gold.aspecto_traducciones
        GROUP BY motor
        ORDER BY num_terminos DESC
    ''', conn)
    grupos_grandes = pd.read_sql('''
        SELECT aspecto_traducido, 
               COUNT(*) AS num_variantes_originales,
               SUM(frecuencia) AS total_menciones,
               STRING_AGG(aspecto_original, ', ' ORDER BY frecuencia DESC) AS variantes
        FROM gold.aspecto_traducciones
        WHERE aspecto_traducido IS NOT NULL
        GROUP BY aspecto_traducido
        ORDER BY total_menciones DESC
        LIMIT 40
    ''', conn)

print('Terminos traducidos por motor:')
display(resumen_motor)

pd.set_option('display.max_colwidth', 200)
display(grupos_grandes)

Terminos traducidos por motor:


,motor,num_terminos
0,None,11083
1,mymemory,5194


,aspecto_traducido,num_variantes_originales,total_menciones,variantes
0,ubicación,105,9955,"location, ubicación, posizione, emplacement, lokalizacja, locatie, localización, ligging, lokalita, lokacija, position, sijainti, localização, beliggenhet, lokalizacji, elhelyezkedés, месторасполо..."
1,departamento,121,9509,"apartment, apartamento, appartement, appartamento, apartament, wohnung, appartment, квартира, mieszkanie, apartments, flat, apartman, апартаменты, apartmán, apartamentos, apartamentai, апартаменти..."
2,vista,38,4054,"view, vistas, views, vue, vista, widok, вид, kilátás, widoki, widokiem, výhled, utsikt, видом, udsigt, výhľad, vistes, zicht, pogled, priveliste, vederea, výhledy, útsýni, priveliște, näköala, wid..."
3,piscina,47,3106,"pool, piscina, piscine, swimming pool, zwembad, basen, бассейн, bazén, басейн, medence, basenu, swimmingpool, baseinas, basenem, uima - allas, basenie, bassein, бассейном, poolanlage, басейну, бас..."
4,anfitrión,41,2627,"host, anfitrión, gastgeber, hôte, gospodarz, szállásadó, gospodarzem, anfitrion, gastheer, anfitrião, házigazda, hostitel, saimniece, värd, gazdă, värden, gospodarza, gostitelj, gastgeberein, gazd..."
5,lugar,43,2156,"place, lugar, miejsce, endroit, posto, lieu, vieta, plek, spot, место, helyen, místo, sted, miejscu, місце, placé, luogo, miejscówka, месте, място, locale, plaats, paikka, místě, sítio, hely, koht..."
6,alojamiento,102,2116,"alojamiento, logement, unterkunft, accommodation, rooms, ubytování, szállás, alloggio, accomodation, помешкання, accommodatie, chambres, ubytovanie, camere, accomodatie, жилье, pokoje, stanze, kam..."
7,casa,19,1694,"casa, house, maison, haus, huis, дом, dům, hús, épület, будинок, ház, hauses, домик, 房 子, kuca, maja, 房 屋, māja, 房"
8,personal,36,1686,"staff, personal, personnel, personale, персонал, personeel, personál, személyzet, personalas, персонала, personalul, staf, osoblje, personelem, personnels, personnelle, henkilökunta, персоналу, pe..."
9,terraza,31,1602,"terraza, terrace, terrasse, terrazza, terrazzino, терраса, terasz, terasse, тераса, tarasem, terasou, terase, terassi, terrass, террасой, verönd, tarasie, terasă, терасою, terrase, 露 台, терасі, te..."


## Notas

- **Ejecutar en local, no en Colab** -- las máquinas de Colab comparten IP de salida, agotando el límite del traductor gratuito mucho más rápido que en una conexión propia.
- **Traductor híbrido**: Google se probó bloqueado (`TooManyRequests`) durante el desarrollo de este notebook. El respaldo con MyMemory permite seguir avanzando aunque Google no responda -- la columna `motor` deja constancia de cuál se usó en cada término, útil para revisar con más atención los que vinieron de MyMemory si hace falta.
- **Limitación conocida del respaldo**: `langdetect` es poco fiable identificando el idioma de términos aislados de 1-2 palabras (a diferencia de frases completas) -- en pruebas reales, falló en 3 de 5 casos de prueba (ej. "tapones" detectado como italiano). Esto afecta principalmente a la traducción de la cola larga de aspectos de baja frecuencia (<10 menciones cada uno); las categorías de mayor peso (ubicación, departamento, piscina...) ya se tradujeron correctamente con Google antes de que se bloqueara, así que el impacto real en el resultado final es marginal.
- Recuerda aplicar después las fusiones manuales de negocio (departamento/apartamento/apartamentos, vista/vistas, ubicación/posición/emplazamiento) y el `ALTER TABLE` + `UPDATE` final sobre `gold.nlp_aspectos_resenas` para rellenar `aspecto_normalizado` con todo lo nuevo.